In [ ]:
from google.colab import drive
import os
import pandas as pd
import numpy as np

drive.mount("/content/drive")

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

PROCESSED_PATH = os.path.join(
    PROJECT_PATH, "data", "processed"
)

OUTPUT_PATH = os.path.join(
    PROJECT_PATH, "outputs"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH, "reports", "tables"
)

os.makedirs(TABLE_PATH, exist_ok=True)

delhi_df = pd.read_parquet(
    os.path.join(
        PROCESSED_PATH,
        "delhi_phase1_checkpoint.parquet"
    )
)

station_metadata = pd.read_parquet(
    os.path.join(
        PROCESSED_PATH,
        "station_metadata_checkpoint.parquet"
    )
)

print("Checkpoint restored successfully.")
print("Delhi records:", len(delhi_df))
print("Delhi stations:", delhi_df["StationId"].nunique())

In [ ]:
delhi_df.info()

In [ ]:
delhi_df["Datetime"] = pd.to_datetime(
    delhi_df["Datetime"],
    errors="coerce"
)

print("Datetime dtype:", delhi_df["Datetime"].dtype)
print("Invalid timestamps:", delhi_df["Datetime"].isna().sum())
print("Start:", delhi_df["Datetime"].min())
print("End:", delhi_df["Datetime"].max())

In [ ]:
duplicate_rows = delhi_df.duplicated().sum()

print("Duplicate rows:", duplicate_rows)

In [ ]:
duplicate_station_time = delhi_df.duplicated(
    subset=["StationId", "Datetime"]
).sum()

print(
    "Duplicate StationId + Datetime records:",
    duplicate_station_time
)

In [ ]:
station_counts = (
    delhi_df
    .groupby(["StationId", "StationName"])
    .size()
    .sort_values(ascending=False)
    .to_frame("RecordCount")
)

display(station_counts)

In [ ]:
station_coverage = (
    delhi_df
    .groupby(["StationId", "StationName"])["Datetime"]
    .agg(
        FirstObservation="min",
        LastObservation="max",
        RecordCount="count"
    )
    .sort_values("RecordCount", ascending=False)
)

display(station_coverage)

In [ ]:
unique_timestamps = (
    delhi_df
    .groupby(["StationId", "StationName"])["Datetime"]
    .nunique()
    .sort_values(ascending=False)
    .to_frame("UniqueTimestamps")
)

display(unique_timestamps)

In [ ]:
sorted_df = delhi_df.sort_values(
    ["StationId", "Datetime"]
).copy()

sorted_df["TimeGap"] = (
    sorted_df
    .groupby("StationId")["Datetime"]
    .diff()
)

time_gaps = (
    sorted_df[
        sorted_df["TimeGap"] > pd.Timedelta(hours=1)
    ]
    .groupby(["StationId", "StationName"])
    .size()
    .sort_values(ascending=False)
    .to_frame("GapsGreaterThan1Hour")
)

display(time_gaps)

In [ ]:
model_columns = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene",
    "Xylene"
]

missing_summary = (
    delhi_df[model_columns]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .to_frame("MissingPercentage")
)

display(missing_summary)

In [ ]:
missing_by_station = (
    delhi_df
    .groupby(["StationId", "StationName"])[model_columns]
    .apply(lambda x: x.isna().mean() * 100)
)

display(missing_by_station)

In [ ]:
aqi_availability = (
    delhi_df
    .groupby(["StationId", "StationName"])["AQI"]
    .agg(
        TotalRecords="size",
        ValidAQI="count"
    )
)

aqi_availability["AQIMissingPercentage"] = (
    1
    - aqi_availability["ValidAQI"]
    / aqi_availability["TotalRecords"]
) * 100

aqi_availability = aqi_availability.sort_values(
    "AQIMissingPercentage"
)

display(aqi_availability)

In [ ]:
negative_values = (
    delhi_df[model_columns]
    .lt(0)
    .sum()
    .sort_values(ascending=False)
    .to_frame("NegativeValueCount")
)

display(negative_values)

In [ ]:
descriptive_stats = (
    delhi_df[model_columns]
    .describe()
    .T
)

display(descriptive_stats)

In [ ]:
aqi_buckets = (
    delhi_df["AQI_Bucket"]
    .value_counts(dropna=False)
)

display(aqi_buckets.to_frame("Count"))

In [ ]:
aqi_bucket_issue = (
    delhi_df["AQI"].notna()
    & delhi_df["AQI_Bucket"].isna()
)

print(
    "AQI present but AQI_Bucket missing:",
    aqi_bucket_issue.sum()
)

In [ ]:
backward_jumps = (
    delhi_df
    .sort_values(["StationId", "Datetime"])
    .groupby("StationId")["Datetime"]
    .apply(
        lambda x: x.diff().lt(
            pd.Timedelta(0)
        ).sum()
    )
    .to_frame("BackwardTimeJumps")
)

display(backward_jumps)

In [ ]:
station_quality = (
    delhi_df
    .groupby(["StationId", "StationName"])
    .agg(
        Records=("Datetime", "size"),
        FirstDate=("Datetime", "min"),
        LastDate=("Datetime", "max"),
        ValidAQI=("AQI", "count"),
        AQIMissingPct=("AQI", lambda x: x.isna().mean() * 100),
        PM25MissingPct=("PM2.5", lambda x: x.isna().mean() * 100),
        PM10MissingPct=("PM10", lambda x: x.isna().mean() * 100),
        NO2MissingPct=("NO2", lambda x: x.isna().mean() * 100)
    )
    .sort_values(
        ["AQIMissingPct", "Records"]
    )
)

display(station_quality)

In [ ]:
station_quality.to_csv(
    os.path.join(
        TABLE_PATH,
        "station_quality_audit.csv"
    )
)

missing_summary.to_csv(
    os.path.join(
        TABLE_PATH,
        "feature_missingness.csv"
    )
)

station_coverage.to_csv(
    os.path.join(
        TABLE_PATH,
        "station_time_coverage.csv"
    )
)

print("Audit tables saved successfully.")
print(TABLE_PATH)

In [ ]:
audit_checkpoint = os.path.join(
    PROCESSED_PATH,
    "delhi_phase2_audit.parquet"
)

delhi_df.to_parquet(
    audit_checkpoint,
    index=False
)

print("Phase 2 checkpoint saved:")
print(audit_checkpoint)

In [ ]:
print("=" * 60)
print("PHASE 2 — DATA QUALITY AUDIT")
print("=" * 60)

print("Delhi records:", len(delhi_df))
print("Delhi stations:", delhi_df["StationId"].nunique())
print("Duplicate rows:", duplicate_rows)
print("Duplicate Station-Time records:", duplicate_station_time)
print("Invalid timestamps:", delhi_df["Datetime"].isna().sum())

print("\nOverall feature missingness:")
display(missing_summary)

print("\nNegative values:")
display(negative_values)

print("=" * 60)

In [ ]:
import os
import json
import pandas as pd

# --------------------------------------------------
# Project paths
# --------------------------------------------------

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

PROCESSED_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "processed"
)

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "outputs"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "tables"
)

os.makedirs(PROCESSED_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(TABLE_PATH, exist_ok=True)


# --------------------------------------------------
# 1. Save Phase 2 working dataset
# --------------------------------------------------

phase2_checkpoint = os.path.join(
    PROCESSED_PATH,
    "delhi_phase2_audit.parquet"
)

delhi_df.to_parquet(
    phase2_checkpoint,
    index=False
)


# --------------------------------------------------
# 2. Save audit tables
# --------------------------------------------------

station_quality.to_csv(
    os.path.join(
        TABLE_PATH,
        "station_quality_audit.csv"
    )
)

missing_summary.to_csv(
    os.path.join(
        TABLE_PATH,
        "feature_missingness.csv"
    )
)

station_coverage.to_csv(
    os.path.join(
        TABLE_PATH,
        "station_time_coverage.csv"
    )

)

aqi_availability.to_csv(
    os.path.join(
        TABLE_PATH,
        "aqi_availability.csv"
    )
)


# --------------------------------------------------
# 3. Save Phase 2 findings
# --------------------------------------------------

phase2_summary = {
    "completed_phase": "Phase 2 - Data Quality Audit",
    "next_phase": "Phase 3 - Data Cleaning & Preprocessing",

    "delhi_records": int(len(delhi_df)),
    "delhi_stations": int(delhi_df["StationId"].nunique()),

    "duplicate_rows": int(duplicate_rows),
    "duplicate_station_time": int(duplicate_station_time),

    "invalid_timestamps": int(
        delhi_df["Datetime"].isna().sum()
    ),

    "datetime_start": str(
        delhi_df["Datetime"].min()
    ),

    "datetime_end": str(
        delhi_df["Datetime"].max()
    ),

    "highest_missing_feature": (
        missing_summary.index[0]
    ),

    "highest_missing_percentage": float(
        missing_summary.iloc[0, 0]
    ),

    "negative_values_total": int(
        negative_values["NegativeValueCount"].sum()
    )
}

summary_file = os.path.join(
    OUTPUT_PATH,
    "phase2_summary.json"
)

with open(summary_file, "w") as f:
    json.dump(
        phase2_summary,
        f,
        indent=4
    )


# --------------------------------------------------
# 4. Verification
# --------------------------------------------------

files_to_check = [
    phase2_checkpoint,
    os.path.join(
        TABLE_PATH,
        "station_quality_audit.csv"
    ),
    os.path.join(
        TABLE_PATH,
        "feature_missingness.csv"
    ),
    os.path.join(
        TABLE_PATH,
        "station_time_coverage.csv"
    ),
    os.path.join(
        TABLE_PATH,
        "aqi_availability.csv"
    ),
    summary_file
]

print("=" * 60)
print("PHASE 2 CHECKPOINT SAVED")
print("=" * 60)

for file in files_to_check:
    print(
        "OK   " if os.path.exists(file) else "ERROR",
        file
    )

print("\nNext phase:")
print("Phase 3 - Data Cleaning & Preprocessing")